[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/01-python-basics/notebooks/03-functions-errors-files.ipynb)

# 01 · Notebook 3 — Functions, errors, tracebacks, files, modules, classes

**Goal:** organise code into reusable pieces, handle things going wrong, and read error messages calmly.

Sections:
1. Functions, parameters, return values, defaults
2. Exceptions and `try` / `except`
3. **Reading tracebacks** (the error report Python prints)
4. Files, `pathlib` and JSON
5. Modules and imports
6. Classes and objects (light)

## 1. Defining a function

`def name(parameters):` starts a **function definition**. The indented block is the function's body. `return` sends a value back. The triple-quoted line is a **docstring** (built-in documentation).

In [1]:
def average(numbers):
    """Return the mean of a list of numbers."""
    total = sum(numbers)
    return total / len(numbers)

print(average([80, 90, 100]))
print(average([1.5, 2.5]))
help(average)

90.0
2.0
Help on function average in module __main__:

average(numbers)
    Return the mean of a list of numbers.



## 2. Defaults and keyword arguments

A **default value** is used when the caller leaves that argument out. **Keyword arguments** (`name=value`) can be given in any order — AI libraries use them everywhere.

In [2]:
def greet(name, greeting="Hello", punctuation="!"):
    return f"{greeting}, {name}{punctuation}"

print(greet("Asha"))
print(greet("Asha", "Namaste"))
print(greet(punctuation="?", name="Ravi"))

Hello, Asha!
Namaste, Asha!
Hello, Ravi?


## 3. Returning several values, and functions without `return`

Returning `a, b` actually returns one tuple, which you can unpack. A function without `return` gives back `None`.

In [3]:
def min_max(values):
    return min(values), max(values)

low, high = min_max([4, 9, 1, 7])
print(low, high)

def say_hi():
    print("hi")

result = say_hi()
print("say_hi returned:", result)

1 9
hi
say_hi returned: None


## 4. Scope: variables inside functions are local

A variable created inside a function exists only while it runs. Pass data **in** with parameters and get it **out** with `return`.

In [4]:
def make_secret():
    secret = 42
    return secret * 2

print(make_secret())
print("'secret' exists outside?", "secret" in globals())

84
'secret' exists outside? False


## 5. Type hints and lambdas (preview)

**Type hints** (`: list[float]`, `-> float`) document expected types; Python does not enforce them, but editors use them. A **lambda** is a tiny one-line function, often used as a `key` for sorting.

In [5]:
def mean(xs: list[float]) -> float:
    return sum(xs) / len(xs)

students = [("Asha", 88), ("Ravi", 92), ("Meera", 75)]
ranked = sorted(students, key=lambda pair: pair[1], reverse=True)
print(mean([1.0, 2.0, 3.0]), ranked)

2.0 [('Ravi', 92), ('Asha', 88), ('Meera', 75)]


**Your turn:** write `letter_grade(score)` returning "A" (90+), "B" (75+), "C" (60+) or "F", and test it on a few scores.

In [6]:
def letter_grade(score):
    if score >= 90:
        return "A"
    if score >= 75:
        return "B"
    if score >= 60:
        return "C"
    return "F"

for s in [95, 80, 61, 30]:
    print(s, letter_grade(s))

95 A
80 B
61 C
30 F


## 6. Exceptions and `try` / `except`

An **exception** is an error that happens while code runs. Without handling, the program stops. `try` / `except` catches it so the program can continue. Catch the **specific** exception you expect.

In [7]:
for text in ["42", "3.5", "abc"]:
    try:
        value = float(text)
        print(f"{text!r} -> {value}")
    except ValueError:
        print(f"{text!r} is not a number, skipping")

'42' -> 42.0
'3.5' -> 3.5
'abc' is not a number, skipping


## 7. The full form: `except ... as err`, `else`, `finally`

- `as err` keeps the error object so you can print its message.
- `else` runs only if **no** exception happened.
- `finally` **always** runs (good for clean-up).

In [8]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  cannot divide by zero")
        return None
    except TypeError as err:
        print("  bad input:", err)
        return None
    else:
        return result
    finally:
        print(f"  finished safe_divide({a!r}, {b!r})")

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "2"))

  finished safe_divide(10, 2)
5.0
  cannot divide by zero
  finished safe_divide(10, 0)
None
  bad input: unsupported operand type(s) for /: 'int' and 'str'
  finished safe_divide(10, '2')
None


## 8. Raising your own exceptions

Use `raise` to reject bad input early, with a clear message. This is better than letting a confusing error appear later.

In [9]:
def set_temperature(t):
    if not 0 <= t <= 2:
        raise ValueError(f"temperature must be between 0 and 2, got {t}")
    return t

try:
    set_temperature(5)
except ValueError as err:
    print("Rejected:", err)

Rejected: temperature must be between 0 and 2, got 5


## 9. Reading tracebacks

When an exception is not caught, Python prints a **traceback** — the chain of function calls that led to the error. To keep this notebook running top to bottom, we catch each error and print its traceback with the standard `traceback` module. It looks exactly like a real crash.

**Read from the bottom up:**
1. **Last line** = *what* went wrong (exception type and message).
2. **The frame above it** = *where* (file or cell, line number, function).
3. **Further up** = *how* the program got there.

In [10]:
import traceback, sys

def show_traceback(func):
    """Run func(); if it fails, print the traceback instead of stopping the notebook."""
    try:
        func()
    except Exception:
        traceback.print_exc(file=sys.stdout)

def average_of(scores):
    total = sum(scores)
    return total / len(scores)

def report(student):
    avg = average_of(student["scores"])
    return f"{student['name']}: {avg:.1f}"

show_traceback(lambda: report({"name": "Ravi", "scores": []}))

Traceback (most recent call last):
  File "/tmp/ipykernel_13363/2670705655.py", line 6, in show_traceback
    func()
    ~~~~^^
  File "/tmp/ipykernel_13363/2670705655.py", line 18, in <lambda>
    show_traceback(lambda: report({"name": "Ravi", "scores": []}))
                           ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_13363/2670705655.py", line 15, in report
    avg = average_of(student["scores"])
  File "/tmp/ipykernel_13363/2670705655.py", line 12, in average_of
    return total / len(scores)
           ~~~~~~^~~~~~~~~~~~~
ZeroDivisionError: division by zero


Reading the traceback above:
- **Last line:** `ZeroDivisionError: division by zero` — we divided by zero.
- **Frame above:** inside `average_of`, the line `return total / len(scores)`. The `~~~^~~~` markers point at the division.
- **Above that:** `report` called `average_of`, and our `lambda` called `report`. (`show_traceback` is our helper; ignore it.)
- **Why:** the scores list was empty, so `len(scores)` was 0.

Now a `KeyError`. Which line failed, and which key was missing?

In [11]:
show_traceback(lambda: report({"name": "Meera"}))

Traceback (most recent call last):
  File "/tmp/ipykernel_13363/2670705655.py", line 6, in show_traceback
    func()
    ~~~~^^
  File "/tmp/ipykernel_13363/2486124851.py", line 1, in <lambda>
    show_traceback(lambda: report({"name": "Meera"}))
                           ~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_13363/2670705655.py", line 15, in report
    avg = average_of(student["scores"])
                     ~~~~~~~^^^^^^^^^^
KeyError: 'scores'


Answer: the last line says `KeyError: 'scores'`; the frame above points at `student["scores"]` inside `report`. The student dict had no `"scores"` key.

Here are several common errors in a row. For each, read only the **last line** and say in your own words what went wrong. Notice the helpful "Did you mean" suggestion for the typo.

In [12]:
examples = {
    "TypeError": lambda: "Score: " + 95,
    "ValueError": lambda: int("ninety"),
    "AttributeError": lambda: "hello".uppr(),
    "IndexError": lambda: [10, 20, 30][3],
    "NameError": lambda: undefined_variable + 1,
}
for name, func in examples.items():
    print(f"--- {name} ---")
    try:
        func()
    except Exception as err:
        # format_exception_only gives exactly the traceback's last line(s), suggestions included
        print("".join(traceback.format_exception_only(err)).strip())

--- TypeError ---
TypeError: can only concatenate str (not "int") to str
--- ValueError ---
ValueError: invalid literal for int() with base 10: 'ninety'
--- AttributeError ---
AttributeError: 'str' object has no attribute 'uppr'. Did you mean: 'upper'?
--- IndexError ---
IndexError: list index out of range
--- NameError ---
NameError: name 'undefined_variable' is not defined


Fixes, in order: convert with `str(95)` or use an f-string; pass a numeric string like `"90"`; spell the method `upper`; the last index of a 3-item list is `2`; define the variable (or run the cell that defines it) first.

**Syntax errors** are different: Python cannot even start. We can see one safely by asking Python to `compile` some broken text. The caret `^` shows where Python got confused.

In [13]:
broken_code = 'print("hello"'
try:
    compile(broken_code, "<example>", "exec")
except SyntaxError as err:
    print(f"SyntaxError: {err.msg} (line {err.lineno})")

SyntaxError: '(' was never closed (line 1)


## 10. Files: write, append, read

Always use `with open(...) as f:` — it closes the file for you. Modes: `"w"` write (replaces), `"a"` append, `"r"` read (the default). Pass `encoding="utf-8"` so every language's characters work. `"\n"` is the newline character.

In [14]:
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("first line\n")
    f.write("second line\n")

with open("notes.txt", "a", encoding="utf-8") as f:
    f.write("third line\n")

with open("notes.txt", encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        print(number, line.strip())

1 first line
2 second line
3 third line


## 11. `pathlib` and a missing file

`pathlib.Path` builds paths that work on Windows, macOS and Linux (`/` joins parts). Opening a file that does not exist raises `FileNotFoundError` — a perfect job for `try` / `except`.

In [15]:
from pathlib import Path

data_dir = Path("practice_data")
data_dir.mkdir(exist_ok=True)
path = data_dir / "scores.txt"
path.write_text("88\n92\n75\n", encoding="utf-8")
numbers = [int(line) for line in path.read_text(encoding="utf-8").splitlines()]
print(path, path.exists(), numbers)

try:
    Path("does_not_exist.txt").read_text()
except FileNotFoundError as err:
    print("Missing file:", err.filename)

practice_data/scores.txt True [88, 92, 75]
Missing file: does_not_exist.txt


## 12. JSON files

**JSON** is the text format APIs use. `json.dump` saves dicts/lists to a file; `json.load` reads them back. `json.dumps` / `json.loads` do the same with strings.

In [16]:
import json

config = {"model": "tiny-model", "temperature": 0.2, "tags": ["demo", "test"]}
json_path = data_dir / "config.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json_path.read_text())
with open(json_path, encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded["tags"][0], type(loaded))

{
  "model": "tiny-model",
  "temperature": 0.2,
  "tags": [
    "demo",
    "test"
  ]
}
demo <class 'dict'>


## 13. Modules and imports

A **module** is a `.py` file of reusable code. The **standard library** ships with Python. `import x`, `from x import y`, and `import x as z` (an **alias**, a nickname) are the three common forms.

In [17]:
import math
from random import choice, seed
import datetime as dt

print(math.sqrt(16), math.pi)
seed(0)                                   # same "random" results every run
print(choice(["rock", "paper", "scissors"]))
print(dt.date(2026, 1, 1).strftime("%A"))

4.0 3.141592653589793
paper
Thursday


**Your own module:** we write a small `helpers.py` file into our practice folder, tell Python to look there, and import from it — exactly how real projects split code across files.

In [18]:
helpers_code = '''
def shout(text):
    return text.upper() + "!"

def word_count(text):
    return len(text.split())
'''
(data_dir / "helpers.py").write_text(helpers_code, encoding="utf-8")

sys.path.insert(0, str(data_dir.resolve()))   # add the folder to Python's search list
from helpers import shout, word_count

print(shout("modules are neat"), word_count("one two three"))

MODULES ARE NEAT! 3


## 14. Classes and objects

A **class** is a blueprint; an **object** is one thing built from it. `__init__` runs when the object is created and sets its **attributes** (its data). **Methods** are functions inside the class; their first parameter `self` means "this object".

In [19]:
class Student:
    def __init__(self, name):
        self.name = name
        self.scores = []

    def add_score(self, score):
        if not 0 <= score <= 100:
            raise ValueError(f"score must be 0-100, got {score}")
        self.scores.append(score)

    def average(self):
        return sum(self.scores) / len(self.scores) if self.scores else 0.0

    def __repr__(self):
        return f"Student({self.name!r}, avg={self.average():.1f})"

asha = Student("Asha")
asha.add_score(88)
asha.add_score(94)
ravi = Student("Ravi")
ravi.add_score(91)
print(asha, ravi)
print(asha.name, asha.scores)

Student('Asha', avg=91.0) Student('Ravi', avg=91.0)
Asha [88, 94]


You will mostly **use** objects others designed: in later modules `df.head()` (pandas), `model.train()` (PyTorch) and `client.messages.create(...)` (an AI SDK) are all method calls on objects. `type()` and `dir()` help you explore an unfamiliar object.

In [20]:
print(type(asha))
print([name for name in dir(asha) if not name.startswith("_")])

<class '__main__.Student'>
['add_score', 'average', 'name', 'scores']


## 15. Clean up

Remove the practice files so the folder stays tidy.

In [21]:
import shutil
Path("notes.txt").unlink(missing_ok=True)
sys.path.remove(str(data_dir.resolve()))
shutil.rmtree(data_dir, ignore_errors=True)
print("cleaned up:", not data_dir.exists())

cleaned up: True


## Summary

- Functions: `def`, parameters, `return`, defaults, keyword arguments; local scope.
- Exceptions: `try` / `except SpecificError as err` / `else` / `finally`; `raise` for bad input.
- Tracebacks: read **bottom up** — what, where, how you got there.
- Files: `with open(..., encoding="utf-8")`, `pathlib.Path`, `json`.
- Modules: `import`, `from ... import`, aliases, your own `.py` files.
- Classes: blueprints with `__init__`, attributes and methods using `self`.

Next: **Notebook 4 — mini-project**.